# MeSA 2.0 — Train YOLOv8n on the medication dataset (VIS-004)

Run on **Google Colab (free GPU)**: Runtime → Change runtime type → **T4 GPU**.

**Goal (M2):** export `best.pt` with **mAP@50 ≥ 0.90** on the val set.

Dataset: 924 images @1280×720, 9 classes — 8 meds (mylanta, vitamin_d3, bayer_aspirin,
cvs_allergy, omeprazole, melatonin, ashwagandha, advil, ~108 each) + `tray` (57, from the
clutter shots). Captured 2026-07-18 at 0.6/0.75/0.9 m — see `docs/capture-protocol.md`.
Note: a few filenames name the wrong med (capture-prompt mixups); Roboflow annotations
are ground truth, never the filename prefix.

Steps: install → pull dataset from Roboflow → train → validate → eval artifacts → download `best.pt`.

In [ ]:
!nvidia-smi
!pip -q install ultralytics==8.2.103 roboflow

In [ ]:
# Pull the annotated dataset (VIS-003).
# Paste YOUR snippet from Roboflow: project page > Export dataset > YOLOv8 > "show download code".
# It looks exactly like this — only the api_key / workspace / project / version differ.
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("mesa-meds")
dataset = project.version(1).download("yolov8")
print(dataset.location)  # contains data.yaml

# Sanity check: 8 med classes + "tray" (labeled in the clutter shots).
import yaml
with open(f"{dataset.location}/data.yaml") as f:
    dy = yaml.safe_load(f)
names = list(dy["names"].values()) if isinstance(dy["names"], dict) else dy["names"]
print(dy["nc"], "classes:", names)
expected = {"mylanta", "vitamin_d3", "bayer_aspirin", "cvs_allergy",
            "omeprazole", "melatonin", "ashwagandha", "advil", "tray"}
assert set(names) == expected, f"class mismatch — wrong project/version? got {names}"

In [ ]:
# v3 ablation: drop the 'tray' class in-notebook (Roboflow's Modify Classes is
# paywalled on the free plan). Deletes tray boxes and remaps class indices above it.
# The tray class caused every group-scene failure in the Day-1 domain-shift results
# (grouped bottles absorbed into one 'tray' box); v3 tests life without it.
import glob, yaml

root = dataset.location
with open(f"{root}/data.yaml") as f:
    dy = yaml.safe_load(f)
names = list(dy["names"].values()) if isinstance(dy["names"], dict) else dy["names"]
if "tray" in names:
    tray_idx = names.index("tray")
    new_names = [n for n in names if n != "tray"]
    changed = removed = 0
    for lbl in glob.glob(f"{root}/*/labels/*.txt"):
        out, dirty = [], False
        for line in open(lbl):
            parts = line.split()
            if not parts:
                continue
            c = int(parts[0])
            if c == tray_idx:
                removed += 1; dirty = True; continue
            if c > tray_idx:
                parts[0] = str(c - 1); dirty = True
            out.append(" ".join(parts))
        if dirty:
            open(lbl, "w").write("\n".join(out) + ("\n" if out else ""))
            changed += 1
    dy["nc"] = len(new_names)
    dy["names"] = new_names
    with open(f"{root}/data.yaml", "w") as f:
        yaml.safe_dump(dy, f)
    print(f"dropped {removed} tray boxes in {changed} files; {dy['nc']} classes: {new_names}")
else:
    print("tray already absent; nothing to do")

In [ ]:
# Synthetic lighting augmentation (domain-shift mitigation arm).
# Evening tungsten light is largely a color-balance + gamma transform of daylight, so we
# TEACH the shift instead of photographing it: every TRAIN image gets a warm and a cool
# variant with labels copied verbatim (geometry unchanged). Val/test stay pristine so
# metrics remain honest. Roughly triples train set; adds ~2 min to training.
import cv2, glob, os, shutil
import numpy as np

VARIANTS = {
    "synwarm": ((1.18, 1.02, 0.72), 1.06),  # tungsten-ish: R up, B down, slight gamma
    "syncool": ((0.82, 0.96, 1.20), 0.96),  # overcast/blue-ish
}
root = dataset.location
made = 0
for f in glob.glob(f"{root}/train/images/*.jpg"):
    base = os.path.basename(f)[:-4]
    if "__syn" in base:
        continue  # idempotent on re-runs
    lbl = f"{root}/train/labels/{base}.txt"
    img = cv2.imread(f).astype(np.float32) / 255.0
    for tag, ((r, g, b), gamma) in VARIANTS.items():
        out = img * np.array([b, g, r])          # BGR order
        out = np.clip(out, 0, 1) ** gamma
        cv2.imwrite(f"{root}/train/images/{base}__{tag}.jpg",
                    (out * 255).astype(np.uint8))
        if os.path.exists(lbl):
            shutil.copy(lbl, f"{root}/train/labels/{base}__{tag}.txt")
        made += 1
print(f"synthetic variants written: {made}")

In [ ]:
# Fine-tune YOLOv8n. 100 epochs w/ early stop is plenty for ~900 imgs / 8 classes.
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,        # early stop if val mAP plateaus
    project="mesa",
    name="yolov8n_med",
)

In [ ]:
# Validate — check mAP@50 >= 0.90 (M2 gate) on the val split.
metrics = model.val()
print("mAP@50   :", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

# Held-out TEST split — this is the honest number for the eval report / paper.
test_metrics = model.val(split="test")
print("TEST mAP@50:", test_metrics.box.map50)

In [ ]:
# Eval artifacts for docs/ (VIS-005): per-class precision/recall table + confusion matrix.
# Download eval_artifacts.zip and unzip into the repo's docs/eval/ folder.
import shutil, os

names = list(dy["names"].values()) if isinstance(dy["names"], dict) else dy["names"]
lines = ["| class | precision | recall | mAP@50 |", "|---|---|---|---|"]
p, r = test_metrics.box.p, test_metrics.box.r
for i, cls_idx in enumerate(test_metrics.box.ap_class_index):
    lines.append(
        f"| {names[int(cls_idx)]} | {p[i]:.3f} | {r[i]:.3f} | {test_metrics.box.ap50[i]:.3f} |"
    )
lines.append(f"\n**Overall test mAP@50: {test_metrics.box.map50:.3f}**")
os.makedirs("eval_out", exist_ok=True)
with open("eval_out/per_class_metrics.md", "w") as f:
    f.write("\n".join(lines))
print("\n".join(lines))

# val() saves confusion_matrix.png etc. into its run dir — grab the latest val run.
runs = sorted([d for d in os.listdir("mesa") if d.startswith("yolov8n_med")], key=lambda d: os.path.getmtime(f"mesa/{d}"))
for f_ in ("confusion_matrix.png", "confusion_matrix_normalized.png"):
    src = f"mesa/{runs[-1]}/{f_}"
    if os.path.exists(src):
        shutil.copy(src, "eval_out/")
shutil.make_archive("eval_artifacts", "zip", "eval_out")
from google.colab import files
files.download("eval_artifacts.zip")

In [ ]:
**After the run:** put `best.pt` at `models/best.pt` on the laptop and Pi (the vision worker
picks it up automatically and switches on detection), and unzip `eval_artifacts.zip` into
`docs/eval/` to fill in `docs/eval-report-template.md` (VIS-005).

If test mAP@50 < 0.90: collect hard negatives (VIS-007), upload them to Roboflow tagged
(e.g. `hard-negatives-aug`), bump the dataset version, and re-run this notebook.